# Probability of KDA
This notebook drafts the analysis of calculating the probability of a given player's KDA statistics any given match interval. Example:
1. What is the probability of a group of X players achieving a mean of 5 `KILLS` at 30 `MINUTES`?

----
# How to Use:

## Import and Working Context
1. Declare working context.
2. Import package dependencies.
3. Import custom local tools.

In [ ]:
%%sql -r dataframe_1
USE WAREHOUSE DV_COMPUTE_WH;

USE DATABASE LEAGUE_RECORDS_LEGACY;

USE SCHEMA L30_ID;


In [ ]:
# ----- Analysis
import pandas as pd
import numpy as np
from scipy.stats import norm
# ----- Visualization Tools
import seaborn as sns
import matplotlib.pyplot as plt
# ----- Vaidations
from enum import StrEnum
from typing import Annotated
from pydantic import (
    BaseModel,
    field_validator,
    model_validator,
    Field
)

In [ ]:
import sys
import os

workspace_root = os.path.dirname(
    os.path.dirname(os.getcwd())
)
sys.path.insert(0, workspace_root)

from analysis.schemas import AnalysisQueryParameters

## Datasets and Query Parameters
* Join KDA dimensions with reports.
* Define and validate user inputs for this draft.

In [ ]:
%%sql -r kda_by_interval
SELECT 
    MINUTE, 
    KILLS, 
    DEATHS, 
    ASSISTS, 
    COALESCE(ROUND(
        (KILLS + ASSISTS) / NULLIF(DEATHS, 0)
    , 2), KILLS + ASSISTS) AS KDA_RATIO
FROM FCT_INTERVALS
JOIN DIM_INTERVALS_KDA USING(PLAYER_INTERVAL_ID)
ORDER BY MINUTE ASC
;

1. `min_interval`: select among valid interval windows provided in the dataset (e.g. 5, 10, 30, ...)
2. `stats`: select among `KILLS`, `DEATHS`, `ASSISTS` or `KDA_RATIO`
3. `bounds`: define the bound value to calculate probability on (e.g. 5 --> 5 `KILLS` at 30 `MINUTES`)

In [ ]:
class KDAStats(StrEnum):
    KILLS = 'KILLS'
    DEATHS = 'DEATHS'
    ASSISTS = 'ASSISTS'
    KDA_RATIO = 'KDA_RATIO'


class ProbabilityOfKDAParams(AnalysisQueryParameters):
    """Parameters for querying KDA probability at a given minute interval. 
    Used to prepare the base dataset and base parameter before analysis starts.
    Example:
        >>> params = ProbabilityOfKDAParams(
        ...     min_interval=10,
        ...     stats='KILLS',
        ...     bounds=10,
        ... )
    """
    min_interval: int
    stats: KDAStats
    bounds: Annotated[int | float, Field(..., ge=0, le=100)]

    @field_validator('min_interval', mode='after')
    @classmethod
    def factor_of_5(cls, v: int) -> int:
        if v % 5 != 0:
            raise ValueError(f'min_interval must be a multiple of 5, got {v}')
        
        return v

    @property
    def query(self) -> str:
        return f"MINUTE == {self.min_interval}"

    def __str__(self) -> str:
        return (
            f"Query: probability of {self.bounds} "
            f"{self.stats} at {self.min_interval} minutes."
        )

In [ ]:
def prep_arr_from_params(
    df: pd.DataFrame, 
    params: ProbabilityOfKDAParams
) -> np.ndarray:
    """Filters and prepares a DataFrame using the given query parameters.
    
    Example:
        >>> prep_arr_from_params(df, ProbabilityOfKDAParams(
        ...     min_interval=10,
        ...     stats='KILLS',
        ...     bounds=10,
        ... ))
    """
    return (
        df
        .query(params.query)
        [params.stats]
        .to_numpy()
    )

## Sampling Distributions
1. Prepare the sampling array as aggregated means across many sample runs of size n.

In [ ]:
def sampling_distribution(
    arr: np.ndarray,
    sample_size: int = 10,
    n_runs: int = 1000,
    seed: int = 42
) -> np.ndarray:
    """Builds a sampling distribution of means from repeated random sampling.

    Example:
        >>> sampling_distribution(arr, sample_size=10, n_runs=1000)
        array([2.3, 1.8, 2.1, ...])
    """
    rng = np.random.default_rng(seed=seed)
    
    samples = rng.choice(
        arr, 
        size=(n_runs, sample_size), # e.g. (1000, 10) matrix
        replace=True
    )
    return samples.mean(axis=1)

## Calculate results
1. Calculate results according to sample space.
2. Convert sample statistics to population statistics and calculate results in population space.

In [ ]:
def calc_ncdf(
    mean: float,
    std: float,
    bound: float,
    round_decimal: int = None,
    reverse_ncdf: bool = False
) -> float:
    """Computes the normal CDF with optional formatting and transformation.

    Args:
        mean: Mean of the normal distribution.
        std: Standard deviation of the normal distribution.
        bound: Value to evaluate the CDF at.
        round_decimal: Number of decimal places to round to. No rounding if None.
        reverse_ncdf: If True, returns the upper tail probability P(X >= bound).

    Example:
        >>> calc_ncdf(mean=3.2, std=1.1, bound=5.0, round_decimal=2)
        0.95
    """
    result = norm.cdf(bound, loc=mean, scale=std)

    if reverse_ncdf:
        result = 1 - result

    if round_decimal is not None:
        result = np.round(result, round_decimal)

    return result

In [ ]:
def convert_std(
    std: float,
    n: int,
    from_space: str,
    to_space: str,
) -> float:
    """Converts a standard deviation between sample and population space.

    Example:
        >>> convert_std(std=2520.87, n=100, from_space='population', to_space='sample')
        252.087
    """
    accept = {'sample', 'population'}
    validate_rules = [
        lambda: from_space not in accept or to_space not in accept,
        lambda: from_space == to_space
    ]
    
    if any(rule() for rule in validate_rules):
        raise ValueError('Invalid arguments! Define between sample and population spaces only!')
    
    if from_space == 'population' and to_space == 'sample':
        return std / np.sqrt(n)
    if from_space == 'sample' and to_space == 'population':
        return std * np.sqrt(n)

## Prepare Reports
1. Using histogram to view singular distribution of the given `stats`.
2. Report and format write ups for notebook presentation.
3. Orchestrate full notebook workflow.

In [ ]:
def visualize_histogram(
    arr: np.ndarray, 
    params: ProbabilityOfKDAParams,
    bins: int,
    **kwargs
) -> None:
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    sns.histplot(
        data=arr,
        ax=ax,
        color='#FF9999',
        kde=True,
        bins=bins,
        **kwargs
    )
    # ----- Labelling 
    ax.set_title(f'Distribution of {params.stats} at Minute {params.min_interval}')
    ax.set_xlabel(params.stats.replace('_', ' ').title())
    ax.set_ylabel('Count')
    # ----- Axis scaling
    ax.set_xlim(0)
    ax.set_ylim(0)
    
    plt.show()
    plt.close(fig)

In [ ]:
def group_probability_of_kda(
    base_dataset: pd.DataFrame,
    min_interval: int,
    stats: str,
    bounds: int,
    sample_size: int = 10,
    n_runs: int = 1000,
) -> dict:
    """Estimate P(group of `sample_size` players averages at least `bounds` stat)
    using sampling distribution + normal CDF (CLT-justified).
    """
    # ----- Validate user inputs and build this run's dataset from query parameters
    params = ProbabilityOfKDAParams(
        min_interval=min_interval,
        stats=stats,
        bounds=bounds,
    )
    arr = prep_arr_from_params(base_dataset, params)

    # ----- Run sampling distribution on stats array
    sampling_arr = sampling_distribution(
        arr=arr,
        sample_size=sample_size,
        n_runs=n_runs
    )

    # ----- Run probability math on sample space (group probability)
    sampling_means = np.mean(sampling_arr)
    sampling_errors = np.std(sampling_arr)
    
    group_probability = calc_ncdf(
        mean=sampling_means,
        std=sampling_errors,
        bound=params.bounds,
        reverse_ncdf=True
    )

    # ----- Visualize the resulting distribution and report
    fmt_group_prob = np.round(group_probability * 100, 2)
    
    visualize_histogram(sampling_arr, params, bins=25)
    
    print(f"Sample mean                  --> {sampling_means:.4f}")
    print(f"Std of sample means (SE)     --> {sampling_errors:.4f}")
    print(
        f"\nAt minute {params.min_interval}:"
        f"\n  Group ({sample_size} players): {fmt_group_prob}% chance of "
        f"averaging at least {params.bounds} {params.stats}."
    )

    return {'group_probability': group_probability}

# Run
----
## How To Use:
1. Set parameters in the function call below.
2. Hit Run All up top if running for first time, or simply rerun the cells on repeated run.

In [ ]:
group_probability_of_kda(
    kda_by_interval,
    # ----- Probability Query
    min_interval=10,
    stats='KILLS',
    bounds=1,
    # ----- Sampling Distribution
    sample_size=100,
    n_runs=10000,
)

In [ ]:
def single_probability_of_kda(
    base_dataset: pd.DataFrame,
    min_interval: int,
    stats: str,
    bounds: int,
) -> dict:
    """Estimate P(a single player achieves at least `bounds` stat)
    using the empirical CDF — no normality assumption.
    """
    # ----- Validate user inputs and build this run's dataset from query parameters
    params = ProbabilityOfKDAParams(
        min_interval=min_interval,
        stats=stats,
        bounds=bounds,
    )
    arr = prep_arr_from_params(base_dataset, params)

    # ----- Empirical single player probability: proportion of observations >= bounds
    single_probability = np.mean(arr >= params.bounds)

    # ----- Report
    fmt_single_prob = np.round(single_probability * 100, 2)

    print(f"Observations (n)             --> {len(arr)}")
    print(f"Mean                         --> {np.mean(arr):.4f}")
    print(f"Std                          --> {np.std(arr):.4f}")
    print(
        f"\nAt minute {params.min_interval} (empirical):"
        f"\n  Single player: {fmt_single_prob}% chance of "
        f"achieving at least {params.bounds} {params.stats}."
    )

    return {'single_probability': single_probability}

In [ ]:
single_probability_of_kda(
    kda_by_interval,
    # ----- Probability Query
    min_interval=40,
    stats='KILLS',
    bounds=10,
)